In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

d:\Semesters\6th_Sem\Minor Project\FairHire\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
job_df = pd.read_csv("../Data/job_description_dataset.csv")
resume_df = pd.read_csv("../Data/resume_dataset.csv")

print("Job Dataset:", job_df.shape)
print("Resume Dataset:", resume_df.shape)

Job Dataset: (1068, 7)
Resume Dataset: (10174, 5)


In [3]:
sample_job = job_df[
    job_df["Title"] == "Data Engineer"
].iloc[0]

sample_resumes = resume_df[
    resume_df["Role"].str.lower() == "data engineer"
].head(10)

print("Selected Job:")
print(sample_job["Title"])

print("\nNumber of Resumes:")
print(len(sample_resumes))

Selected Job:
Data Engineer

Number of Resumes:
10


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("SBERT Model Loaded Successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1522.80it/s]


SBERT Model Loaded Successfully!


In [5]:
job_text = (
    sample_job["Title"] + " " +
    sample_job["Skills"] + " " +
    sample_job["Responsibilities"] + " " +
    sample_job["Keywords"]
)

resume_texts = sample_resumes["Resume"].tolist()

job_embedding = model.encode(job_text)

resume_embeddings = model.encode(resume_texts)

print("Job Embedding Shape:", job_embedding.shape)
print("Resume Embedding Shape:", resume_embeddings.shape)

Job Embedding Shape: (384,)
Resume Embedding Shape: (10, 384)


In [6]:
semantic_scores = cosine_similarity(
    [job_embedding],
    resume_embeddings
)[0]

semantic_df = pd.DataFrame({
    "Resume_ID": sample_resumes.index,
    "Semantic_Score": semantic_scores
})

semantic_df["Semantic_Percentage"] = (
    semantic_df["Semantic_Score"] * 100
).round(2)

semantic_df = semantic_df.sort_values(
    by="Semantic_Percentage",
    ascending=False
)

semantic_df

,Resume_ID,Semantic_Score,Semantic_Percentage
9,196,0.649941,64.989998
7,179,0.617691,61.770000
0,22,0.606810,60.680000
6,173,0.603875,60.389999
5,155,0.574880,57.490002
2,54,0.567418,56.740002
4,142,0.548787,54.880001
1,49,0.540547,54.049999
3,141,0.533085,53.310001
8,195,0.517908,51.790001


In [7]:
required_skills = [
    skill.strip().lower()
    for skill in sample_job["Keywords"].split(";")
]

print("Required Skills")
print("-" * 40)

for skill in required_skills:
    print(skill)

Required Skills
----------------------------------------
python
sql
etl
big data
aws
hadoop
mongodb
data engineering fundamentals


In [8]:
top_resume_id = semantic_df.iloc[0]["Resume_ID"]

top_resume = sample_resumes.loc[top_resume_id, "Resume"]

print("Top Resume ID:", top_resume_id)

print("\nResume Preview")
print("=" * 50)

print(top_resume[:1200])

Top Resume ID: 196.0

Resume Preview
Here's a sample professional resume for Tamara Wells:

Tamara Wells
Contact Information:

* Email: [tamara.wells@email.com](mailto:tamara.wells@email.com)
* Phone: 555-555-5555
* LinkedIn: linkedin.com/in/tamara-wells
* GitHub: github.com/tamara-wells

Professional Summary:
Highly motivated and experienced Data Engineer with a proven track record of designing and implementing large-scale data pipelines and architectures on cloud platforms. Skilled in MLOps, ETL, Big Data, and Spark, with a strong passion for leveraging data to drive business insights and innovation.

Technical Skills:

* Programming languages: Python, Java, Scala
* Cloud platforms: AWS, Azure, Google Cloud
* Big Data technologies: Hadoop, Spark, Hive, Pig
* Data warehousing: Amazon Redshift, Google BigQuery
* ETL tools: Apache NiFi, Talend, Informatica PowerCenter
* MLOps frameworks: TensorFlow, PyTorch, scikit-learn
* Agile methodologies: Scrum, Kanban
* Operating systems: Linux, W

In [9]:
matched_skills = []

resume_text = top_resume.lower()

for skill in required_skills:
    if skill in resume_text:
        matched_skills.append(skill)

print("Matched Skills")
print("-" * 40)

for skill in matched_skills:
    print("✔", skill)

Matched Skills
----------------------------------------
✔ python
✔ etl
✔ big data
✔ aws
✔ hadoop


In [10]:
missing_skills = []

for skill in required_skills:
    if skill not in resume_text:
        missing_skills.append(skill)

print("Missing Skills")
print("-" * 40)

for skill in missing_skills:
    print("✘", skill)

Missing Skills
----------------------------------------
✘ sql
✘ mongodb
✘ data engineering fundamentals


In [11]:
skill_match_percentage = (
    len(matched_skills) / len(required_skills)
) * 100

print("Matched Skills :", len(matched_skills))
print("Total Skills   :", len(required_skills))
print("Skill Match    :", round(skill_match_percentage, 2), "%")

Matched Skills : 5
Total Skills   : 8
Skill Match    : 62.5 %


In [12]:
semantic_percentage = semantic_df.iloc[0]["Semantic_Percentage"]

print("=" * 60)
print("FINAL RESUME EXPLANATION")
print("=" * 60)

print(f"\nResume ID : {int(top_resume_id)}")

print(f"Semantic Score : {semantic_percentage:.2f}%")
print(f"Skill Match    : {skill_match_percentage:.2f}%")

print("\nMatched Skills")
print("-" * 30)

for skill in matched_skills:
    print("✔", skill)

print("\nMissing Skills")
print("-" * 30)

for skill in missing_skills:
    print("✘", skill)

FINAL RESUME EXPLANATION

Resume ID : 196
Semantic Score : 64.99%
Skill Match    : 62.50%

Matched Skills
------------------------------
✔ python
✔ etl
✔ big data
✔ aws
✔ hadoop

Missing Skills
------------------------------
✘ sql
✘ mongodb
✘ data engineering fundamentals


In [13]:
# ==========================================================
# FAIRHIRE AI ASSESSMENT REPORT
# ==========================================================

semantic_percentage = semantic_df.iloc[0]["Semantic_Percentage"]

final_score = (
    semantic_percentage * 0.70 +
    skill_match_percentage * 0.30
)

print("="*90)
print("                 FAIRHIRE AI ASSESSMENT REPORT")
print("="*90)

# ----------------------------------------------------------
# Candidate Summary
# ----------------------------------------------------------

print("\n1. Candidate Summary")
print("-"*90)

print(f"Resume ID            : {int(top_resume_id)}")
print(f"Job Position         : {sample_job['Title']}")
print(f"Semantic Score       : {semantic_percentage:.2f}%")
print(f"Skill Match Score    : {skill_match_percentage:.2f}%")
print(f"Final Hybrid Score   : {final_score:.2f}%")

# ----------------------------------------------------------
# Semantic Analysis
# ----------------------------------------------------------

print("\n2. Semantic Understanding")
print("-"*90)

print("SBERT converts both the Job Description and Resume into")
print("384-dimensional embeddings.")

print("Cosine Similarity measures how closely the overall")
print("meaning of the resume matches the job description.")

print(f"\nSemantic Similarity Obtained : {semantic_percentage:.2f}%")

# ----------------------------------------------------------
# Skill Analysis
# ----------------------------------------------------------

print("\n3. Technical Skill Analysis")
print("-"*90)

print(f"Required Skills : {len(required_skills)}")
print(f"Matched Skills  : {len(matched_skills)}")
print(f"Missing Skills  : {len(missing_skills)}")

print("\nMatched Skills")

if matched_skills:
    for skill in matched_skills:
        print(f"   ✔ {skill}")
else:
    print("   None")

print("\nMissing Skills")

if missing_skills:
    for skill in missing_skills:
        print(f"   ✘ {skill}")
else:
    print("   None")

# ----------------------------------------------------------
# Strengths
# ----------------------------------------------------------

print("\n4. Candidate Strengths")
print("-"*90)

if semantic_percentage >= 70:
    print("✔ Excellent semantic alignment with the job description.")
elif semantic_percentage >= 50:
    print("✔ Good semantic alignment with the job description.")
else:
    print("✔ Partial semantic alignment with the job description.")

if len(matched_skills) > 0:
    print(f"✔ Demonstrates {len(matched_skills)} required technical skills.")

print("✔ Resume contains relevant professional experience.")

# ----------------------------------------------------------
# Areas for Improvement
# ----------------------------------------------------------

print("\n5. Areas for Improvement")
print("-"*90)

if len(missing_skills) == 0:
    print("No missing technical skills were identified.")
else:
    for skill in missing_skills:
        print(f"• Improve knowledge of {skill}")

# ----------------------------------------------------------
# Hybrid Scoring
# ----------------------------------------------------------

print("\n6. Hybrid Scoring Breakdown")
print("-"*90)

semantic_contribution = semantic_percentage * 0.70
skill_contribution = skill_match_percentage * 0.30

print(f"Semantic Contribution (70%) : {semantic_contribution:.2f}")
print(f"Skill Contribution (30%)    : {skill_contribution:.2f}")

print("-----------------------------------------------")
print(f"Final Hybrid Score          : {final_score:.2f}%")

# ----------------------------------------------------------
# AI Recommendation
# ----------------------------------------------------------

print("\n7. AI Hiring Recommendation")
print("-"*90)

if final_score >= 80:
    recommendation = "Highly Recommended"
    confidence = "Very High"

elif final_score >= 65:
    recommendation = "Recommended"
    confidence = "High"

elif final_score >= 50:
    recommendation = "Consider for Interview"
    confidence = "Moderate"

else:
    recommendation = "Not Recommended"
    confidence = "Low"

print(f"Recommendation : {recommendation}")
print(f"Confidence     : {confidence}")

# ----------------------------------------------------------
# Transparency
# ----------------------------------------------------------

print("\n8. Explainability & Fairness")
print("-"*90)

print("This recommendation was generated using only:")

print("✔ Resume Content")
print("✔ Job Description")
print("✔ Semantic Similarity")
print("✔ Technical Skill Matching")

print("\nThe following attributes were NOT considered:")

print("✘ Candidate Name")
print("✘ Gender")
print("✘ Age")
print("✘ Email Address")
print("✘ Phone Number")
print("✘ Home Address")
print("✘ Nationality")

print("\nThe decision is transparent because every score")
print("can be traced back to semantic similarity and")
print("explicitly matched technical skills.")

print("="*90)

                 FAIRHIRE AI ASSESSMENT REPORT

1. Candidate Summary
------------------------------------------------------------------------------------------
Resume ID            : 196
Job Position         : Data Engineer
Semantic Score       : 64.99%
Skill Match Score    : 62.50%
Final Hybrid Score   : 64.24%

2. Semantic Understanding
------------------------------------------------------------------------------------------
SBERT converts both the Job Description and Resume into
384-dimensional embeddings.
Cosine Similarity measures how closely the overall
meaning of the resume matches the job description.

Semantic Similarity Obtained : 64.99%

3. Technical Skill Analysis
------------------------------------------------------------------------------------------
Required Skills : 8
Matched Skills  : 5
Missing Skills  : 3

Matched Skills
   ✔ python
   ✔ etl
   ✔ big data
   ✔ aws
   ✔ hadoop

Missing Skills
   ✘ sql
   ✘ mongodb
   ✘ data engineering fundamentals

4. Candidate Stre

In [14]:
print("="*90)
print("                 FAIRHIRE EXECUTIVE SUMMARY")
print("="*90)

print(f"""
Candidate ID           : {int(top_resume_id)}
Applied Position       : {sample_job['Title']}

Overall Match Score    : {final_score:.2f}%

Decision               : {recommendation}

Summary
-------
The candidate was evaluated using FairHire's Hybrid AI
Ranking Framework, which combines:

• Semantic Similarity (SBERT)
• Technical Skill Matching

The final recommendation was generated using a weighted
evaluation of semantic relevance (70%) and technical
skill matching (30%).

This evaluation is explainable, transparent and does
not consider personal demographic information.

End of Report.
""")

print("="*90)

                 FAIRHIRE EXECUTIVE SUMMARY

Candidate ID           : 196
Applied Position       : Data Engineer

Overall Match Score    : 64.24%

Decision               : Consider for Interview

Summary
-------
The candidate was evaluated using FairHire's Hybrid AI
Ranking Framework, which combines:

• Semantic Similarity (SBERT)
• Technical Skill Matching

The final recommendation was generated using a weighted
evaluation of semantic relevance (70%) and technical
skill matching (30%).

This evaluation is explainable, transparent and does
not consider personal demographic information.

End of Report.

